In [0]:
'''
1. create a workspace client
2. mention the workspace and notebook paths
3. associate each notebook to a job task and configure dependent job if any
4. check if the job exists with the job name
5. create the job with job name and aggregates job tasks(in a list)
6. if the pipeline exists run it with the new settings
7. if it does not exists create a new pipeline
8. run the job with run_now command with the job id
9. check the run id via the job_id_runner.response.run_id
'''
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient()

WORKSPACE_BASE_PATH = "/Workspace/Users/abhirajadhikary06@gmail.com/payment-gateway-databricks-pipeline"

notebook_paths = {
    "setup": f"{WORKSPACE_BASE_PATH}/00_setup_tables",
    "data-generation": f"{WORKSPACE_BASE_PATH}/01_data_generation",
    "bronze-transform": f"{WORKSPACE_BASE_PATH}/02_bronze_transform",
    "silver-transform": f"{WORKSPACE_BASE_PATH}/03_silver_transform",
    "gold-transform": f"{WORKSPACE_BASE_PATH}/04_gold_transform",
    "ai-agent": f"{WORKSPACE_BASE_PATH}/05_ai_agent",
}

task_setup = jobs.Task(
    task_key="00_setup_tables",
    notebook_task=jobs.NotebookTask(notebook_path=notebook_paths["setup"])
)

task_gen = jobs.Task(
    task_key="01_data_generation",
    depends_on=[jobs.TaskDependency(task_key="00_setup_tables")],
    notebook_task=jobs.NotebookTask(notebook_path=notebook_paths["data-generation"])
)

task_bronze = jobs.Task(
    task_key="02_bronze_transform",
    depends_on=[jobs.TaskDependency(task_key="01_data_generation")],
    notebook_task=jobs.NotebookTask(notebook_path=notebook_paths["bronze-transform"])
)

task_silver = jobs.Task(
    task_key="03_silver_transform",
    depends_on=[jobs.TaskDependency(task_key="02_bronze_transform")],
    notebook_task=jobs.NotebookTask(notebook_path=notebook_paths["silver-transform"])
)

task_gold = jobs.Task(
    task_key="04_gold_transform",
    depends_on=[jobs.TaskDependency(task_key="03_silver_transform")],
    notebook_task=jobs.NotebookTask(notebook_path=notebook_paths["gold-transform"])
)

task_ai = jobs.Task(
    task_key="05_ai_agent",
    depends_on=[jobs.TaskDependency(task_key="04_gold_transform")],
    notebook_task=jobs.NotebookTask(notebook_path=notebook_paths["ai-agent"])
)

cron_schedule = jobs.CronSchedule(
    quartz_cron_expression="0 0 12 ? * MON-FRI *",
    timezone_id="UTC",
    pause_status=jobs.PauseStatus.UNPAUSED
)

JOB_NAME = "payment-gateway-pipeline"
existing_job_id = None

for job in w.jobs.list(name=JOB_NAME):
    if job.settings.name == JOB_NAME:
        existing_job_id = job.job_id
        break 

if existing_job_id:
    w.jobs.reset(
        job_id=existing_job_id,
        new_settings=jobs.JobSettings(
            name=JOB_NAME,
            tasks=[task_setup, task_gen, task_bronze, task_silver, task_gold, task_ai],
            schedule=cron_schedule
        )
    )
    job_id = existing_job_id
    print(f"Workflow Job Updated Successfully! (Job ID: {job_id})")
else:
    created_job = w.jobs.create(
        name=JOB_NAME,
        tasks=[task_setup, task_gen, task_bronze, task_silver, task_gold, task_ai],
        schedule=cron_schedule
    )
    job_id = created_job.job_id
    print(f"Workflow Job Created Successfully! (Job ID: {job_id})")

# Updated to use job_id instead of created_job.job_id
run_now_response = w.jobs.run_now(job_id=job_id)
run_id = run_now_response.response.run_id

print(f"Pipeline Run Triggered Successfully! Run ID: {run_id}")

In [0]:
import time
run_details = w.jobs.get_run(run_id=run_id)
print(f"Tracking Tasks for Run ID: {run_id}\n")

for task in run_details.tasks:
    task_name = task.task_key
    state = task.state.life_cycle_state.value
    result = task.state.result_state.value if task.state.result_state else "RUNNING"
    print(f"Task: {task_name:<25} | State: {state:<12} | Result: {result}")